# DNABERT-2 + SupCon: первый baseline на CPU
Замораживаем DNABERT-2, обучаем общую MLP и сравниваем поиск mouse-партнёров **до и после** на одном VISTA holdout.

**VISTA-supported SCREEN retrieval:** human ELS пересекает позитивный нативный hg38 VISTA-интервал ≥1 bp; правильные mouse-партнёры берутся из SCREEN. Это не независимая VISTA-разметка ортологии. Ткани и стадии VISTA не сопоставляем с SCREEN.

Используем `active_pairs_with_dna.parquet` после предварительного tissue-фильтра: DNase и H3K27ac > 1.64 у обоих видов.
Это тканевые прокси-позитивы: максимумы разных assays могли быть получены из разных биообразцов.
`pospos_K1` создаёт **негативные соответствия между ELS**, а не доказанные не-энхансерные последовательности.

Для оригинального кода модели фиксируем Transformers 4.44.2, загрузку без meta-инициализации и обычный PyTorch attention.
Уже рассчитанные батчи эмбеддингов сохраняются в `data/cache/dnabert2_embeddings` и переиспользуются при повторном запуске.
Backbone загружается из исходной фиксированной revision с `trust_remote_code=True`.

Локально проверены структура/синтаксис блокнота и логика кэша на тестовых данных. Полный запуск требует модулей `eclare`,
ваших Parquet/BED/VISTA и весов модели; в этой среде эти данные отсутствуют.


## Один раз: зависимости загрузчика DNABERT-2
Выполни следующую ячейку, затем **Restart Kernel** и запускай блокнот начиная с импортов.
Эта установка не заменяет зависимости проекта из `requirements-baseline.txt` и установленный PyTorch.


In [ ]:
%pip install "transformers==4.44.2" einops


## 1. Импорты и параметры

In [ ]:
from pathlib import Path
from datetime import datetime
from math import comb
import hashlib
import importlib
import json
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, BertConfig

print("Transformers:", transformers.__version__, "| PyTorch:", torch.__version__)
if transformers.__version__ != "4.44.2":
    raise RuntimeError("Установи Transformers 4.44.2 в первой ячейке и перезапусти kernel")


In [ ]:
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'eclare').is_dir():
    ROOT = Path(r"C:\Projects\eclare")
sys.path.insert(0, str(ROOT))

from eclare.data.negatives import build_pospos_dataset
from eclare.data.vista import load_vista, overlap_vista, VISTA_COMMIT

DATA = ROOT / 'data'
OUT = DATA / 'experiments' / datetime.now().strftime('baseline_%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True)
EMBED_CACHE = DATA / 'cache/dnabert2_embeddings'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)


In [ ]:
CFG = dict(seed=42, epochs=3, batch_size=256, lr=1e-3, weight_decay=1e-4,
           temperature=0.1, head_hidden=256, embedding_dim=128, encode_batch_size=4)
MODEL = 'zhihan1996/DNABERT-2-117M'
REVISION = '7bce263b15377fc15361f52cfab88f8b586abda0'
KS = [1, 5, 10, 20]
DEVICE = 'cpu'

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
print('Устройство:', DEVICE)
pd.Series(CFG, name='значение').to_frame()


## 2. Позитивы
Каждая пара повторяется по тканям. Для sampling оставляем строки, для DNA — уникальные элементы. Модель получает только DNA, без assay/tissue-признаков. Координаты: human hg38 и mouse mm10, BED (0-based half-open). N не удаляем.

In [ ]:
POSITIVES_PATH = DATA / 'processed/active_pairs_with_dna.parquet'
if not POSITIVES_PATH.is_file():
    raise FileNotFoundError(f"Сначала сохрани тканевые позитивы в {POSITIVES_PATH}")
positives = pd.read_parquet(POSITIVES_PATH)
signals = ['human_zscore_DNase', 'human_zscore_H3K27ac',
           'mouse_zscore_DNase', 'mouse_zscore_H3K27ac']
if positives.empty or not positives[signals].gt(1.64).all(axis=1).all():
    raise ValueError('Ожидаются непустые тканевые позитивы: все четыре сигнала > 1.64')
pairs = positives[['human_id', 'mouse_id']].drop_duplicates()
print('Строк:', len(positives), '| пар:', len(pairs), '| тканей:', positives.tissue.nunique())
positives.head()


In [ ]:
elements = {}
for species in ['human', 'mouse']:
    columns = [f'{species}_{c}' for c in ['id', 'chrom', 'start', 'end', 'sequence']]
    elements[species] = positives[columns].drop_duplicates().reset_index(drop=True)
    print(species, '| элементов:', len(elements[species]))
    display(elements[species].head())


In [ ]:
homologs = pd.read_csv(
    DATA / 'encode/hg38-mm10-Homologous.tsv', sep='	', header=None,
    usecols=[0, 1], names=['human_id', 'mouse_id'],
).drop_duplicates()
available_pairs = homologs[
    homologs.human_id.isin(elements['human'].human_id) &
    homologs.mouse_id.isin(elements['mouse'].mouse_id)
]
available_pairs.head()


## 3. VISTA: пересечение ≥1 bp
[Источник](https://gitlab.com/egsb-mfgl/vista-data): фиксированный `locus.tsv.gz`. Loader использует нативные `assembly/coord`, не межвидовой liftOver; переводит start−1, end не меняет. `positive` — активность хотя бы в одном из проверенных условий, не доказательство активности всего пересекающегося ELS.

In [ ]:
vista = load_vista(DATA / 'external' / f'vista-{VISTA_COMMIT}')
display(vista.head())
vista.groupby(['assembly', 'curation_status']).size().rename('локусы').to_frame()


In [ ]:
hits = {}
for species in ['human', 'mouse']:
    native = vista[vista.species.eq(species)]
    hits[species] = overlap_vista(elements[species], native, species=species)
    print(species, '| VISTA-hit ELS:', hits[species][f'{species}_id'].nunique())
    display(hits[species].head())


In [ ]:
vista_positive = vista[(vista.species == 'human') & (vista.curation_status == 'positive')]
validation_hits = hits['human'][hits['human'].vista_id.isin(vista_positive.vista_id)]
val_human_ids = set(validation_hits.human_id)
val_pairs = available_pairs[available_pairs.human_id.isin(val_human_ids)]
print('VISTA локусов:', validation_hits.vista_id.nunique(), '| human:', len(val_human_ids), '| пар:', len(val_pairs))
val_pairs.head()


## 4. Train / validation без общих локусов
В holdout включаем все VISTA-tested ELS обоих видов, связанные homolog-компоненты, пересекающиеся интервалы и точные DNA-дубликаты. Расширяем до неподвижной точки. Это прежний консервативный split; компоненты определяют только split, а не позитивные метки. Почти одинаковые DNA не кластеризуются.

In [ ]:
held = {s: set(hits[s][f'{s}_id']) for s in ['human', 'mouse']}
while True:
    previous_size = sum(map(len, held.values()))
    linked = homologs.human_id.isin(held['human']) | homologs.mouse_id.isin(held['mouse'])
    for species in ['human', 'mouse']:
        held[species].update(homologs.loc[linked, f'{species}_id'])
        table = elements[species]
        selected = table[table[f'{species}_id'].isin(held[species])]
        same_dna = table[f'{species}_sequence'].isin(selected[f'{species}_sequence'])
        held[species].update(table.loc[same_dna, f'{species}_id'])
        intervals = selected.rename(columns={f'{species}_{c}': c for c in ['chrom', 'start', 'end']})
        intervals = intervals.rename(columns={f'{species}_id': 'vista_id'})
        overlaps = overlap_vista(table, intervals, species=species, progress=False)
        held[species].update(overlaps[f'{species}_id'])
    if sum(map(len, held.values())) == previous_size:
        break


In [ ]:
train_positive = positives[
    ~positives.human_id.isin(held['human']) & ~positives.mouse_id.isin(held['mouse'])
].reset_index(drop=True)
queries = elements['human'][elements['human'].human_id.isin(val_human_ids)].sort_values('human_id')
candidates = elements['mouse'][elements['mouse'].mouse_id.isin(held['mouse'])].sort_values('mouse_id')
print('Train:', len(train_positive), '| исключено:', len(positives) - len(train_positive))
print('Validation запросов:', len(queries), '| mouse-кандидатов:', len(candidates))
train_positive.head()

if train_positive.empty or queries.empty or candidates.empty or val_pairs.empty:
    raise ValueError('Пустой train или VISTA holdout: сначала проверь покрытие текущей выборки')


## 5. Негативы pospos_K1
Один негатив на позитивную строку: human сохраняем, mouse выбираем среди уникальных IDs той же ткани, исключая всех известных позитивных партнёров. DNA/признаки mouse меняются вместе. Без GC/length matching; повторы негативных пар разрешены.

In [ ]:
train_data = build_pospos_dataset(train_positive, seed=CFG['seed'])
display(train_data.head())
train_data.groupby(['label', 'negative_type']).size().rename('строки').to_frame()


## 6. Frozen DNABERT-2
Обычная DNA без k-mer пробелов и усечения. Mean pooling исключает padding, сохраняет специальные токены. Backbone всегда в eval, градиенты считаем только для MLP.
На CPU начинаем с encode batch size 4. Предварительный прогон трёх последовательностей ниже покажет,
работает ли загрузка. При повторе кэш ускорит preview, поэтому это не полноценный benchmark.
Для повторного запуска после прерывания можно выполнить блокнот сначала: готовые батчи не пересчитываются.
Ключ кэша включает MODEL, REVISION, правило pooling и точные последовательности батча.
Изменение encode batch size изменит разбиение на батчи и потребует пересчёта соответствующих файлов.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
config = BertConfig.from_pretrained(MODEL, revision=REVISION)
backbone = AutoModel.from_pretrained(
    MODEL, revision=REVISION, trust_remote_code=True, config=config,
    low_cpu_mem_usage=False, device_map=None, torch_dtype=torch.float32,
).to(DEVICE).eval()
backbone.requires_grad_(False)


In [ ]:
# Отключаем Triton-путь для CPU в исходном модуле загруженной модели.
layers = importlib.import_module(backbone.__class__.__module__)
layers.flash_attn_qkvpacked_func = None
print('Backbone:', next(backbone.parameters()).device)


In [ ]:
@torch.inference_mode()
def encode_batch(batch):
    tokens = tokenizer(batch, padding=True, truncation=False, return_tensors='pt').to(DEVICE)
    hidden = backbone(**tokens)[0]
    mask = tokens['attention_mask'].unsqueeze(-1)
    return ((hidden * mask).sum(1) / mask.sum(1)).cpu().float()


In [ ]:
def cached_encode_batch(batch):
    signature = dict(model=MODEL, revision=REVISION,
                     pooling='mean-no-padding-with-specials-fp32-v1', sequences=batch)
    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
    path = EMBED_CACHE / f'{key}.pt'
    if path.is_file():
        return torch.load(path, map_location='cpu', weights_only=True)
    result = encode_batch(batch)
    temporary = path.with_suffix('.tmp')
    torch.save(result, temporary)
    temporary.replace(path)
    return result


In [ ]:
def encode_dna(sequences):
    if not sequences:
        raise ValueError('Нет последовательностей для DNABERT-2')
    blocks = []
    for start in tqdm(range(0, len(sequences), CFG['encode_batch_size']), desc='DNABERT-2 / cache'):
        batch = sequences[start:start + CFG['encode_batch_size']]
        blocks.append(cached_encode_batch(batch))
    return torch.cat(blocks)


In [ ]:
from time import perf_counter

started = perf_counter()
preview = encode_dna(elements['human'].human_sequence.head(3).tolist())
print('Форма:', preview.shape, '| секунд:', round(perf_counter() - started, 2))
pd.DataFrame(preview.numpy()).iloc[:, :6].head()


In [ ]:
features, id_to_index = {}, {}
for species, table in elements.items():
    sequences = table[f'{species}_sequence']
    unique_dna = sequences.drop_duplicates().tolist()
    dna_index = {dna: i for i, dna in enumerate(unique_dna)}
    features[species] = encode_dna(unique_dna)[sequences.map(dna_index).to_numpy()]
    id_to_index[species] = {element: i for i, element in enumerate(table[f'{species}_id'])}
    print(species, features[species].shape)
    display(pd.DataFrame(features[species].numpy()).iloc[:, :6].head())
del backbone


## 7. Retrieval-метрики
Сортируем один фиксированный holdout mouse-пул по cosine similarity. Все прямые SCREEN-партнёры запроса — правильные. **Recall@k** — доля правильных ответов в top-k; **hit@k** — найден хотя бы один; **MRR** — обратное место первого правильного. Усредняем по human-запросам. При одном правильном ответе recall и hit совпадают. Равные scores упорядочены по mouse ID.

In [ ]:
query_indices = queries.human_id.map(id_to_index['human']).to_numpy()
candidate_indices = candidates.mouse_id.map(id_to_index['mouse']).to_numpy()
candidate_position = {mouse: i for i, mouse in enumerate(candidates.mouse_id)}
truth = val_pairs.groupby('human_id').mouse_id.agg(set).to_dict()
queries.head()


In [ ]:
def evaluate_retrieval(human_features, mouse_features):
    q = F.normalize(human_features[query_indices], dim=1)
    c = F.normalize(mouse_features[candidate_indices], dim=1)
    rows = []
    for start in tqdm(range(0, len(q), 64), desc='Retrieval'):
        scores = (q[start:start + 64] @ c.T).numpy()
        for human, score in zip(queries.human_id.iloc[start:start + 64], scores):
            order = np.argsort(-score, kind='stable')
            relevant = [candidate_position[m] for m in truth[human]]
            ranks = np.sort(np.argsort(order)[relevant] + 1)
            row = dict(human_id=human, positive_ranks=ranks.tolist(), mrr=1 / ranks[0])
            for k in KS:
                row[f'recall@{k}'] = (ranks <= k).mean()
                row[f'hit@{k}'] = (ranks <= k).any()
            rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def summarize(ranks, stage):
    n = len(candidates)
    relevant_counts = ranks.positive_ranks.map(len)
    return pd.DataFrame([dict(
        stage=stage, k=k, recall=ranks[f'recall@{k}'].mean(),
        hit=ranks[f'hit@{k}'].mean(), mrr=ranks.mrr.mean(),
        random_recall=min(k, n) / n,
        random_hit=np.mean([1 - comb(n - r, min(k, n)) / comb(n, min(k, n)) for r in relevant_counts]),
    ) for k in KS])


## 8. VISTA до обучения
Исходные 768D DNABERT-2 эмбеддинги, без MLP.

In [ ]:
before_ranks = evaluate_retrieval(features['human'], features['mouse'])
before_metrics = summarize(before_ranks, 'before')
display(before_ranks.head())
before_metrics


## 9. MLP + SupCon
768 → 256 → GELU → 128 → L2, без dropout. AdamW, 3 эпохи, без scheduler и early stopping. VISTA не участвует в выборе checkpoint.

Batch содержит positive human/mouse и K1 negative mouse. Каждый ID входит один раз; все прямые train-партнёры в batch — позитивы, остальные cross-species сочетания — негативы. Same-species сравнений нет. Поэтому эффективных негативов в loss больше одного. Mouse без позитива в batch остаётся кандидатом, но не anchor. Неизвестные связи могут оказаться false negatives. Тканевые повторы сохраняют вес строк, но ткань модель не получает.

In [ ]:
positive_rows = train_data[train_data.label.eq(1)].sort_values(['human_id', 'tissue'], kind='stable')
negative_rows = train_data[train_data.label.eq(0)].sort_values(['human_id', 'tissue'], kind='stable')
dataset = TensorDataset(
    torch.tensor(positive_rows.human_id.map(id_to_index['human']).to_numpy()),
    torch.tensor(positive_rows.mouse_id.map(id_to_index['mouse']).to_numpy()),
    torch.tensor(negative_rows.mouse_id.map(id_to_index['mouse']).to_numpy()),
)
loader = DataLoader(dataset, batch_size=CFG['batch_size'], shuffle=True,
                    generator=torch.Generator().manual_seed(CFG['seed']))
train_partners = {}
for human, mouse in train_positive[['human_id', 'mouse_id']].drop_duplicates().itertuples(index=False, name=None):
    train_partners.setdefault(id_to_index['human'][human], set()).add(id_to_index['mouse'][mouse])


In [ ]:
torch.manual_seed(CFG['seed'])
head = nn.Sequential(
    nn.Linear(features['human'].shape[1], CFG['head_hidden']), nn.GELU(),
    nn.Linear(CFG['head_hidden'], CFG['embedding_dim']),
).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
head


In [ ]:
def supcon_batch(batch):
    human, positive_mouse, negative_mouse = batch
    human = human.unique()
    mouse = torch.cat([positive_mouse, negative_mouse]).unique()
    mask = torch.tensor([[m in train_partners[h] for m in mouse.tolist()]
                         for h in human.tolist()], device=DEVICE)
    zh = F.normalize(head(features['human'][human].to(DEVICE)), dim=1)
    zm = F.normalize(head(features['mouse'][mouse].to(DEVICE)), dim=1)
    logits = zh @ zm.T / CFG['temperature']
    # Усредняем log-probability всех позитивов в обоих направлениях.
    human_logprob = F.log_softmax(logits, dim=1)
    mouse_logprob = F.log_softmax(logits, dim=0)
    human_loss = -(human_logprob * mask).sum(1) / mask.sum(1)
    valid_mouse = mask.any(0)
    mouse_loss = -(mouse_logprob * mask).sum(0)[valid_mouse] / mask.sum(0)[valid_mouse]
    return (human_loss.mean() + mouse_loss.mean()) / 2


In [ ]:
# Один batch, без обновления весов.
loss = supcon_batch(tuple(t[:8] for t in dataset.tensors))
loss.backward()
print('Loss:', loss.item(), '| gradient:', head[0].weight.grad.norm().item())
optimizer.zero_grad()


In [ ]:
history = []
for epoch in range(CFG['epochs']):
    head.train()
    losses = []
    bar = tqdm(loader, desc=f'Эпоха {epoch + 1}')
    for batch in bar:
        optimizer.zero_grad()
        loss = supcon_batch(batch)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        bar.set_postfix(loss=f'{loss.item():.4f}')
    history.append(dict(epoch=epoch + 1, loss=np.mean(losses)))
pd.DataFrame(history)


## 10. VISTA после обучения
Те же запросы, mouse-пул и метки. Финальная голова после трёх эпох. Сравнение raw 768D → trained 128D включает и обучение, и изменение размерности.

In [ ]:
head.eval()
projected = {}
with torch.no_grad():
    for species, values in features.items():
        blocks = [F.normalize(head(block.to(DEVICE)), dim=1).cpu()
                  for block in tqdm(values.split(CFG['batch_size']), desc=f'MLP: {species}')]
        projected[species] = torch.cat(blocks)


In [ ]:
after_ranks = evaluate_retrieval(projected['human'], projected['mouse'])
after_metrics = summarize(after_ranks, 'after')
metrics = pd.concat([before_metrics, after_metrics], ignore_index=True)
display(after_ranks.head())
metrics.pivot(index='k', columns='stage', values=['recall', 'hit', 'mrr'])


## 11. Recall@k: до → после
Одна панель на k, общая шкала, пунктир — случайный поиск. Один seed, без error bars.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.linewidth': 0.5, 'pdf.fonttype': 42})


In [ ]:
fig, axes = plt.subplots(1, len(KS), figsize=(7.2, 2.6), sharey=True)
for ax, k in zip(axes, KS):
    scores = metrics[metrics.k.eq(k)].set_index('stage')
    values = scores.loc[['before', 'after'], 'recall'].to_numpy()
    bars = ax.bar(['До', 'После'], values, width=0.45, color=['#B8B8B8', '#2A78D6'], zorder=3)
    bars[0].set_hatch('///')
    ax.axhline(scores.loc['before', 'random_recall'], color='0.4', ls='--', lw=0.8)
    ax.set(title=f'Recall@{k}', ylim=(0, 1.08))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', color='0.92', linewidth=0.5, zorder=0)
    ax.bar_label(bars, labels=[f'{v:.1%}' for v in values], padding=3, fontsize=8)
axes[0].set_ylabel('Recall')
fig.legend(bars, ['DNABERT-2', 'DNABERT-2 + SupCon MLP'],
           loc='upper center', ncol=2, frameon=False)
fig.text(0.5, 0.01, f'VISTA-supported SCREEN · {len(queries)} запросов · {len(candidates)} кандидатов',
         ha='center', fontsize=8, color='0.4')
fig.tight_layout(rect=(0, 0.07, 1, 0.9))
fig.savefig(OUT / 'recall_before_after.pdf', bbox_inches='tight')
fig.savefig(OUT / 'recall_before_after.png', dpi=300, bbox_inches='tight')
plt.show()


## 12. Сохранить результат
Оставляем параметры, split, sampled negatives, ранги, метрики и финальную голову. Без сбора версий, hashes и большого manifest. Seed фиксирован, но побитовая детерминированность CUDA не принуждается.
Замороженные DNA-эмбеддинги уже сохранены по батчам в общем каталоге кэша.


In [ ]:
train_data.to_parquet(OUT / 'train_pospos_K1.parquet', index=False)
val_pairs.to_parquet(OUT / 'validation_pairs.parquet', index=False)
validation_hits.to_parquet(OUT / 'vista_hits.parquet', index=False)
queries[['human_id']].to_csv(OUT / 'queries.csv', index=False)
candidates[['mouse_id']].to_csv(OUT / 'candidates.csv', index=False)
before_ranks.to_parquet(OUT / 'ranks_before.parquet', index=False)
after_ranks.to_parquet(OUT / 'ranks_after.parquet', index=False)
metrics.to_csv(OUT / 'metrics.csv', index=False)
pd.DataFrame(history).to_csv(OUT / 'history.csv', index=False)
pd.Series(dict(CFG, model=MODEL, revision=REVISION, vista_commit=VISTA_COMMIT, ks=KS)).to_json(OUT / 'config.json', indent=2)
torch.save(head.state_dict(), OUT / 'head_final.pt')
print('Результаты:', OUT)


### Ограничения
Один seed; первые 20 000 human IDs исходного subset — не случайная выборка. VISTA подтверждает активность human-фрагмента, не пары; overlap ≥1 bp — слабая поддержка. Почти идентичные DNA могут пересекать split, а reference DNA могла встречаться в pretraining DNABERT-2. Повторный подбор параметров по VISTA превратит её в development set. Метрики и график появятся только после запуска блокнота.